In [1]:
!python --version


Python 3.13.5


In [1]:
import os

PROJECT_FOLDERS = [
    "MP_Data",
    "frames",
    "landmarks",
    "models",
    "scripts",
    "outputs"
]

SIGNS = [
    "Hello", "ThankYou", "Sorry", "Please", "Yes", "No",
    "Stop", "Go", "Come", "Help",
    "Water", "Food", "Washroom",
    "Happy", "Sad"
]

# Create project folders
for folder in PROJECT_FOLDERS:
    os.makedirs(folder, exist_ok=True)

# Create sign folders
for sign in SIGNS:
    os.makedirs(os.path.join("MP_Data", sign), exist_ok=True)
    os.makedirs(os.path.join("frames", sign), exist_ok=True)

print("✅ FULL Hand2Voice project folder structure created")


✅ FULL Hand2Voice project folder structure created


In [3]:
## Important this is the recording script here we changed signs name again
import cv2
import os
import time

# ================= CONFIG =================
BASE_DIR = r"C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data"

SIGN_NAME = "Time"        # ← CHANGE THIS EACH TIME
VIDEOS_TO_RECORD = 25

PREP_TIME = 3
RECORD_TIME = 2
FPS = 20
CAMERA_INDEX = 0
FRAME_SIZE = (640, 480)
# =========================================

cap = cv2.VideoCapture(CAMERA_INDEX)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_SIZE[0])
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_SIZE[1])

sign_dir = os.path.join(BASE_DIR, SIGN_NAME)
os.makedirs(sign_dir, exist_ok=True)

existing = [f for f in os.listdir(sign_dir) if f.endswith(".mp4")]
start_idx = len(existing) + 1

fourcc = cv2.VideoWriter_fourcc(*'mp4v')

print(f"Recording sign: {SIGN_NAME}")
print(f"Starting from video index: {start_idx}")

for i in range(start_idx, start_idx + VIDEOS_TO_RECORD):

    # ---------- PREP ----------
    for sec in range(PREP_TIME, 0, -1):
        ret, frame = cap.read()
        cv2.putText(frame, f"{SIGN_NAME} | Get Ready: {sec}",
                    (40, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 255, 0), 2)
        cv2.imshow("Hand2Voice Recorder", frame)
        cv2.waitKey(1000)

    # ---------- RECORD ----------
    output_path = os.path.join(sign_dir, f"{SIGN_NAME}_{i}.mp4")
    out = cv2.VideoWriter(output_path, fourcc, FPS, FRAME_SIZE)

    start_time = time.time()
    while time.time() - start_time < RECORD_TIME:
        ret, frame = cap.read()
        cv2.putText(frame, f"{SIGN_NAME} | Recording {i}",
                    (40, 50), cv2.FONT_HERSHEY_SIMPLEX,
                    1, (0, 0, 255), 2)
        out.write(frame)
        cv2.imshow("Hand2Voice Recorder", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    out.release()
    print(f"Saved: {output_path}")

cap.release()
cv2.destroyAllWindows()
print("✅ Recording complete")


Recording sign: Please
Starting from video index: 21
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_21.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_22.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_23.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_24.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_25.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_26.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_27.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_28.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data\Please\Please_29.mp4
Saved: C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Dat

In [ ]:
## “Landmarks extraction"

import cv2
import os
import numpy as np
import mediapipe as mp

# ================= CONFIG =================
BASE_DIR = r"C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data"
SEQUENCE_LENGTH = 30
# =========================================

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

def extract_landmarks(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(frame_rgb)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]
            landmarks = [[lm.x, lm.y, lm.z] for lm in hand.landmark]
            frames.append(landmarks)

    cap.release()

    if len(frames) < SEQUENCE_LENGTH:
        return None

    return np.array(frames[:SEQUENCE_LENGTH])

# ========= MAIN EXTRACTION LOOP =========
for sign in os.listdir(BASE_DIR):
    sign_path = os.path.join(BASE_DIR, sign)
    if not os.path.isdir(sign_path):
        continue

    print(f"\nProcessing sign: {sign}")
    file_index = 0

    for file in os.listdir(sign_path):
        if not file.endswith(".mp4"):
            continue

        video_path = os.path.join(sign_path, file)
        landmarks = extract_landmarks(video_path)

        if landmarks is None:
            print(f"Skipped (too short / no hand): {file}")
            continue

        npy_path = os.path.join(sign_path, f"{file_index}.npy")
        np.save(npy_path, landmarks)
        print(f"Saved: {npy_path}")
        file_index += 1

hands.close()
print("\n✅ ALL LANDMARKS EXTRACTED")


In [ ]:
##🔥 STEP 1 — COUNT .npy FILES (RUN THIS)

import os

BASE_DIR = r"C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data"

print("NPY COUNT PER SIGN:\n")

for sign in os.listdir(BASE_DIR):
    sign_path = os.path.join(BASE_DIR, sign)
    if not os.path.isdir(sign_path):
        continue

    npy_count = len([f for f in os.listdir(sign_path) if f.endswith(".npy")])
    print(f"{sign:12s} : {npy_count}")


In [ ]:
# updated landmark for only three signs
import cv2
import os
import numpy as np
import mediapipe as mp

# ================= CONFIG =================
BASE_DIR = r"C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data"
TARGET_SIGNS = ["Happy", "Finish", "Please"]
SEQUENCE_LENGTH = 30
# =========================================

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.6,
    min_tracking_confidence=0.6
)

def extract_landmarks(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(frame_rgb)

        if result.multi_hand_landmarks:
            hand = result.multi_hand_landmarks[0]
            landmarks = [[lm.x, lm.y, lm.z] for lm in hand.landmark]
            frames.append(landmarks)

    cap.release()

    if len(frames) < SEQUENCE_LENGTH:
        return None

    return np.array(frames[:SEQUENCE_LENGTH])

# ========= MAIN LOOP =========
for sign in TARGET_SIGNS:
    sign_path = os.path.join(BASE_DIR, sign)

    print(f"\nProcessing sign: {sign}")

    # Delete old npy files (important!)
    for f in os.listdir(sign_path):
        if f.endswith(".npy"):
            os.remove(os.path.join(sign_path, f))

    file_index = 0

    for file in os.listdir(sign_path):
        if not file.endswith(".mp4"):
            continue

        video_path = os.path.join(sign_path, file)
        landmarks = extract_landmarks(video_path)

        if landmarks is None:
            print(f"Skipped: {file}")
            continue

        npy_path = os.path.join(sign_path, f"{file_index}.npy")
        np.save(npy_path, landmarks)
        file_index += 1

    print(f"✅ Saved {file_index} landmark files for {sign}")

hands.close()
print("\n🎉 Critical signs updated successfully!")


Skipped: Please_7.mp4
Skipped: Please_8.mp4


In [2]:
#Confirm dataset shape 
import os
import numpy as np

BASE_DIR = r"C:\Users\diyav\OneDrive\Desktop\Hand2Voice\anaconda_projects\MP_Data"

X = []
y = []
signs = []

# Collect valid signs (only folders with enough .npy files)
for sign in os.listdir(BASE_DIR):
    sign_path = os.path.join(BASE_DIR, sign)
    if not os.path.isdir(sign_path):
        continue

    npy_files = [f for f in os.listdir(sign_path) if f.endswith(".npy")]

    # Only include signs with enough data
    if len(npy_files) >= 8:
        signs.append(sign)

# Sort for consistent label order
signs = sorted(signs)

label_map = {sign: idx for idx, sign in enumerate(signs)}

for sign in signs:
    sign_path = os.path.join(BASE_DIR, sign)
    for file in os.listdir(sign_path):
        if file.endswith(".npy"):
            data = np.load(os.path.join(sign_path, file))
            X.append(data)
            y.append(label_map[sign])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of classes:", len(signs))
print("Classes:", signs)

X shape: (346, 30, 21, 3)
y shape: (346,)
Number of classes: 21
Classes: ['Come', 'Doctor', 'Emergency', 'Food', 'Go', 'Good', 'Hello', 'Help', 'Home', 'Namaste', 'No', 'Pain', 'Sad', 'Sleep', 'Sorry', 'Stop', 'ThankYou', 'Washroom', 'Welcome', 'Where', 'Yes']


In [3]:
X = X.reshape(X.shape[0], X.shape[1], -1)
print("Reshaped X:", X.shape)

Reshaped X: (346, 30, 63)


In [4]:
#train and validation split
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# One-hot encode labels
y_encoded = to_categorical(y)

X_train, X_val, y_train, y_val = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Train shape: (276, 30, 63)
Val shape: (70, 30, 63)


In [5]:
#LSTM model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

model = Sequential()

model.add(LSTM(128, return_sequences=True, input_shape=(30, 63)))
model.add(Dropout(0.3))

model.add(LSTM(64))
model.add(Dropout(0.3))

model.add(Dense(64, activation='relu'))
model.add(Dense(len(signs), activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 30, 128)           98304     
                                                                 
 dropout (Dropout)           (None, 30, 128)           0         
                                                                 
 lstm_1 (LSTM)               (None, 64)                49408     
                                                                 
 dropout_1 (Dropout)         (None, 64)                0         
                                                                 
 dense (Dense)               (None, 64)                4160      
                                                                 
 dense_1 (Dense)             (None, 21)                1365      
                                                                 
Total params: 153237 (598.58 KB)
Trainable params: 153

In [6]:
#actual training 
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=16,
    callbacks=[early_stop]
)

Epoch 1/50


18/18 [==============================] - 18s 303ms/step - loss: 3.0352 - accuracy: 0.0688 - val_loss: 2.9576 - val_accuracy: 0.1286
Epoch 2/50
18/18 [==============================] - 1s 84ms/step - loss: 2.9581 - accuracy: 0.1123 - val_loss: 2.7620 - val_accuracy: 0.2000
Epoch 3/50
18/18 [==============================] - 2s 87ms/step - loss: 2.7591 - accuracy: 0.1558 - val_loss: 2.5025 - val_accuracy: 0.2286
Epoch 4/50
18/18 [==============================] - 2s 87ms/step - loss: 2.5068 - accuracy: 0.1413 - val_loss: 2.3539 - val_accuracy: 0.2429
Epoch 5/50
18/18 [==============================] - 2s 88ms/step - loss: 2.2682 - accuracy: 0.2645 - val_loss: 2.0512 - val_accuracy: 0.3000
Epoch 6/50
18/18 [==============================] - 2s 93ms/step - loss: 2.0847 - accuracy: 0.3080 - val_loss: 1.8247 - val_accuracy: 0.4429
Epoch 7/50
18/18 [==============================] - 2s 89ms/step - loss: 1.8138 - accuracy: 0.4312 - val_loss: 1.5631 - val_accuracy: 0.5429
Epoch 8/5

In [7]:
#evaluation 
val_loss, val_acc = model.evaluate(X_val, y_val)
print("Validation Accuracy:", val_acc)

3/3 [==============================] - 0s 25ms/step - loss: 0.3242 - accuracy: 0.9143
Validation Accuracy: 0.9142857193946838


In [8]:
model.save("hand2voice_model.h5")

C:\Users\diyav\anaconda3\envs\hand2voice\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [10]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)

# 🔥 IMPORTANT FIX
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open("hand2voice_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

INFO:tensorflow:Assets written to: C:\Users\diyav\AppData\Local\Temp\tmpgghri82q\assets


INFO:tensorflow:Assets written to: C:\Users\diyav\AppData\Local\Temp\tmpgghri82q\assets


TFLite model saved successfully!
